# Gold Business Layer - Olist Brazilian E-commerce

Conceptual model locked: `Retail_Gold_Model.md`  
Bronze/Silver complete - not redesigned. Gold contains exactly 6 tables:

**Facts:** `fact_orders` (1 row/order), `fact_order_items` (1 row/order_item), `fact_payments` (1 row/payment)  
**Dimensions:** `dim_customer` (1 row/customer_id), `dim_product` (1 row/product), `dim_seller` (1 row/seller)  

**Excluded from Gold:** `reviews`, `geolocation`  
**Principles:** grain before design, separate facts by grain (avoid 3x2=6 fan-out), `customer_unique_id` = analytical identity, `customer_id` retained for traceability, timestamps remain timestamps, lowercase snake_case, no derived metrics in first build.

Implementation sequence: Load Silver -> dim_customer -> dim_product -> dim_seller -> fact_orders -> fact_order_items -> fact_payments -> Validate

In [0]:
# -------------------------------------------
# Project Configuration
# -------------------------------------------

CATALOG = "retail_demo"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# Gold tables are written as Delta managed tables in Unity Catalog
# Consistent with 01_env_setup.ipynb and 03_silver_transformation.ipynb conventions

In [0]:
# Ensure Gold schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA}").show(truncate=False)

In [0]:
# -------------------------------------------
# Load required Silver tables
# -------------------------------------------
from pyspark.sql.functions import col

customers_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers")
orders_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.orders")
order_items_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.order_items")
products_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.products")
sellers_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.sellers")
payments_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.payments")
category_translation_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.product_category_translation")

# Quick row-count sanity vs Silver (should reconcile with Gold later)
for name, df in [
    ("customers", customers_silver),
    ("orders", orders_silver),
    ("order_items", order_items_silver),
    ("products", products_silver),
    ("sellers", sellers_silver),
    ("payments", payments_silver),
    ("product_category_translation", category_translation_silver),
]:
    print(f"{name:<30} rows={df.count():>12,}  cols={len(df.columns)}")
#For loop has name and df, because when we print, Python may not know or doesn't know what df is, so we need to give it a name.
#The dataframe does not knwo which sticker we put in so we need a name tag here
#Now the name:<30 means the name will be left aligned and take upto 30 characters and then rows will start and take upto 12 character and finally columns will be printed.
#We have rows=df.count():>12, ;comma is used to selerate thousands for readability.

## 1. dim_customer
Grain: 1 row per `customer_id` (PK `customer_id`), business identity `customer_unique_id` (1 unique_id -> N customer_id).  
Source: `silver.customers` — no join required. Gold is business-facing, drops `ingestion_timestamp`/`source_file`.  
Columns: `customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state`

In [0]:
from pyspark.sql.functions import col

# dim_customer: select Gold columns, preserve Silver types (zip = int)
dim_customer = customers_silver.select(
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix"),
    col("customer_city"),
    col("customer_state")
)

# Validate grain before write
#I understand the two print statements where the first one is used to print the row count for dim_customer
#Second one is used to count the dictinct customer_id and count the number and seperate it with thousands using comma for readability
#Third statement is looping through two columns, customer_id and customer_unique_id and
#for each row, it checks if the value is NULL, it returns true or false and cast turns it into int that is 1 or 0
#and then we alias the column with the same name and finally we show the results (note show() only prints first 20 rows by default)
#If that row has a null value for any column it will show as 1, if not then 0.

print(f"dim_customer rows: {dim_customer.count():,}")
print(f"distinct customer_id: {dim_customer.select('customer_id').distinct().count():,}")
dim_customer.select([col(c).isNull().cast("int").alias(c) for c in ["customer_id","customer_unique_id"]]).show()

In [0]:
# Write dim_customer to Gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

(
    dim_customer.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")
)
#Format is delta means we will have parquet files in the background and we can use delta lake features like time travel, schema evolution, etc.

dim_customer_final = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")
dim_customer_final.printSchema() #prints the schema of the table that we stored in the gold layer. It will show the column names and their data types.
display(dim_customer_final.limit(10))
print(f"Gold dim_customer rows: {dim_customer_final.count():,}") #prints rows and we can manually check the silver rows and gold rows.

## 2. dim_product
Grain: 1 row per `product_id` (PK `product_id`). Enriches `product_category_name` with English translation via LEFT JOIN (keeps 620 null-category products).  
Source: `silver.products` LEFT JOIN `silver.product_category_translation` on `product_category_name`.  
No separate category dimension. Corrects Silver spelling `lenght -> length` already handled in Silver.

In [0]:
from pyspark.sql.functions import col

# dim_product: LEFT JOIN to preserve products with null category
dim_product = (
    products_silver.alias("p")
    .join(
        category_translation_silver.alias("t"),
        on=col("p.product_category_name") == col("t.product_category_name"),
        how="left"
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name"),
        col("t.product_category_name_english"),
        col("p.product_name_length"),
        col("p.product_description_length"),
        col("p.product_photos_qty"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm")
    )
)
#We are enriching the silver table products by adding a new column from the category_translation_silver
#We are doing it here because it makes sense to add a single column in the table rather than creating a new dim_category_translation_silver and implement an extra join
#We are doing a left join emphasing more on the products table rather than category_translation_silver table
#We may get null values and that should be ok for now and we will not remove them as of now, but definetly we need to check this

print(f"dim_product rows: {dim_product.count():,}")
#We will print the row count for dim_product and we also have this "," to seprate thousands for readability
print(f"distinct product_id: {dim_product.select('product_id').distinct().count():,}")
#Here we are printing the distinct count of product_id, note we distinct created a new dataframe of product_id and then we coount the rows from this dataframe
# Check translation coverage
# dim_product.select(col("product_category_name_english").isNull().cast("int").alias("null_english")).groupBy().sum().show()
#This one is quite crazy as we are trying to see how many null values we have in the specific column called product_category_name_english
#All we want to check is the total number of null values in this column and I believe we can do it via filter as well instead of groupby().sum().show()
#So we can go with filter and count rows which is much simpler to understand and filter is a partition-local operation.
dim_product.filter(col("product_category_name_english").isNull()).count()
# I am going with this approach which is much simpler and it reduces one extra step of casting.


In [0]:
(
    dim_product.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_product")
)

#This one is basic, writing the dim_product table in delta format and then we overwrite whenever we run the code
#Then we saveAsTable to save it in a specific catalog and schema in the gold layer.
dim_product_final = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_product")
dim_product_final.printSchema()
display(dim_product_final.limit(10))
print(f"Gold dim_product rows: {dim_product_final.count():,}")
print(f"Bronze/Silver products rows should reconcile: Silver={products_silver.count():,} Gold={dim_product_final.count():,}")

## 3. dim_seller
Grain: 1 row per `seller_id` (PK `seller_id`).  
Source: `silver.sellers` — no join. Includes regional attributes required by Gold (no separate geography dim).

In [0]:
dim_seller = sellers_silver.select(
    col("seller_id"),
    col("seller_zip_code_prefix"),
    col("seller_city"),
    col("seller_state")
)

#This cell has the row count print statement and distinct count print as well and remember distinct creates a new df and then we count the rows of that dataframe
#Lastly we have used a for loop for one column called seller_id which is not optimal i guess
print(f"dim_seller rows: {dim_seller.count():,}")
print(f"distinct seller_id: {dim_seller.select('seller_id').distinct().count():,}")
# dim_seller.select([col(c).isNull().cast("int").alias(c) for c in ["seller_id"]]).show()
null_seller_ids = dim_seller.filter(
    col("seller_id").isNull()
).count()
#I would rather prefer a filter query which is easy to understand and does the same job

In [0]:
(
    dim_seller.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_seller")
)

dim_seller_final = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_seller")
dim_seller_final.printSchema()
display(dim_seller_final.limit(10))
#Same as we did earlier, where we write the table in delta format and then we have overwrite it.

## 4. fact_orders
Grain: 1 row per `order_id` (PK `order_id`).  
Source: `silver.orders` + `silver.customers` to resolve `customer_unique_id` (analytical identity) — `silver.orders` only holds `customer_id`.  
Joins: `orders.customer_id = customers.customer_id` LEFT JOIN (preserve orders even if customer lookup fails for FK validation).  
Timestamps remain timestamps; `order_estimated_delivery_date` cast from Silver `date` to Gold `timestamp` per locked schema. No derived metrics.

In [0]:
from pyspark.sql.functions import col

# fact_orders: enrich customer_unique_id via customers lookup
# Select Gold columns in locked order; cast estimated date -> timestamp
fact_orders = (
    orders_silver.alias("o")
    .join(customers_silver.alias("c"), on="customer_id", how="left")
    .select(
        col("o.order_id"),
        col("c.customer_unique_id"),
        col("o.customer_id"),
        col("o.order_status"),
        col("o.order_purchase_timestamp"),
        col("o.order_approved_at"),
        col("o.order_delivered_carrier_date"),
        col("o.order_delivered_customer_date"),
        col("o.order_estimated_delivery_date").cast("timestamp").alias("order_estimated_delivery_date")
    )
)

print(f"fact_orders rows: {fact_orders.count():,}")
print(f"distinct order_id: {fact_orders.select('order_id').distinct().count():,}")
# FK check: should be 0 orphan customer_unique_id if Silver is consistent
fact_orders.select(col("customer_unique_id").isNull().cast("int").alias("null_unique_id")).groupBy().sum().show()
fact_orders.printSchema()

In [0]:
(
    fact_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
)

fact_orders_final = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
fact_orders_final.printSchema()
display(fact_orders_final.limit(10))
print(f"Bronze/Silver orders vs Gold fact_orders: Silver={orders_silver.count():,} Gold={fact_orders_final.count():,}")

## 5. fact_order_items
Grain: 1 row per `(order_id, order_item_id)` (composite key).  
Source: `silver.order_items` — no joins at build time to avoid fact-fan-out (3 items x 2 payments = 6 rows if joined). Relationships to `dim_product`/`dim_seller` are via FK at query time.  
Columns: `order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value` (price/freight already decimal(10,2) in Silver).

In [0]:
fact_order_items = order_items_silver.select(
    col("order_id"),
    col("order_item_id"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date"),
    col("price"),
    col("freight_value")
)

print(f"fact_order_items rows: {fact_order_items.count():,}")
print(f"distinct (order_id, order_item_id): {fact_order_items.select('order_id','order_item_id').distinct().count():,}")
fact_order_items.printSchema()

In [0]:
(
    fact_order_items.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")
)

fact_order_items_final = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")
fact_order_items_final.printSchema()
display(fact_order_items_final.limit(10))
print(f"Silver order_items vs Gold: Silver={order_items_silver.count():,} Gold={fact_order_items_final.count():,}")

## 6. fact_payments
Grain: 1 row per `(order_id, payment_sequential)` (composite key).  
Source: `silver.payments` — no joins. Keep separate from `fact_orders`/`fact_order_items` to preserve additive measures (payment_value).  
Columns: `order_id, payment_sequential, payment_type, payment_installments, payment_value`

In [0]:
fact_payments = payments_silver.select(
    col("order_id"),
    col("payment_sequential"),
    col("payment_type"),
    col("payment_installments"),
    col("payment_value")
)

print(f"fact_payments rows: {fact_payments.count():,}")
print(f"distinct (order_id, payment_sequential): {fact_payments.select('order_id','payment_sequential').distinct().count():,}")
fact_payments.printSchema()

In [0]:
(
    fact_payments.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_payments")
)

fact_payments_final = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_payments")
fact_payments_final.printSchema()
display(fact_payments_final.limit(10))
print(f"Silver payments vs Gold: Silver={payments_silver.count():,} Gold={fact_payments_final.count():,}")

## 7. Gold Validation
Per `Retail_Gold_Model.md` #13: confirm grain, key uniqueness, expected row counts, no fan-out, FK resolution, nulls, totals reconcile.  
Excluded tables intentionally not created: `fact_reviews`, `dim_geography`.

In [0]:
# Validation 1: Row counts & expected reconciliation with Silver
gold_tables = ["dim_customer", "dim_product", "dim_seller", "fact_orders", "fact_order_items", "fact_payments"]
for t in gold_tables:
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{t}")
    print(f"{t:<20} rows={df.count():>8,}  cols={len(df.columns):>2}  schema={', '.join([f'{f.name}:{f.dataType.simpleString()}' for f in df.schema.fields])}")

# Detailed Silver vs Gold reconciliation
print("\n=== Silver vs Gold row-count reconciliation ===")
pairs = [
    ("customers", customers_silver, spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")),
    ("products", products_silver, spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_product")),
    ("sellers", sellers_silver, spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_seller")),
    ("orders", orders_silver, spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")),
    ("order_items", order_items_silver, spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")),
    ("payments", payments_silver, spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_payments")),
]
for name, s, g in pairs:
    print(f"{name:<15} Silver={s.count():>8,} Gold={g.count():>8,} delta={g.count()-s.count():>6,}")

In [0]:
# Validation 2: Grain / PK uniqueness (should be 0 duplicates)
from pyspark.sql.functions import col

checks = [
    ("dim_customer PK customer_id", f"{CATALOG}.{GOLD_SCHEMA}.dim_customer", ["customer_id"]),
    ("dim_product PK product_id", f"{CATALOG}.{GOLD_SCHEMA}.dim_product", ["product_id"]),
    ("dim_seller PK seller_id", f"{CATALOG}.{GOLD_SCHEMA}.dim_seller", ["seller_id"]),
    ("fact_orders PK order_id", f"{CATALOG}.{GOLD_SCHEMA}.fact_orders", ["order_id"]),
    ("fact_order_items PK (order_id,order_item_id)", f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items", ["order_id","order_item_id"]),
    ("fact_payments PK (order_id,payment_sequential)", f"{CATALOG}.{GOLD_SCHEMA}.fact_payments", ["order_id","payment_sequential"]),
]
for label, table, keys in checks:
    df = spark.table(table)
    dup = df.count() - df.select(*keys).distinct().count()
    print(f"{label:<50} duplicates={dup:,}  {'PASS' if dup==0 else 'FAIL'}")

In [0]:
# Validation 3: Foreign-key resolution & no unexpected nulls in keys
# dim_customer -> fact_orders (customer_id) 1:many, fact_orders -> fact_order_items/payments (order_id)
from pyspark.sql.functions import col

dim_customer = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")
dim_product = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_product")
dim_seller = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_seller")
fact_orders = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
fact_order_items = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")
fact_payments = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_payments")

# Nulls in business keys (should be 0)
for label, df, keys in [
    ("dim_customer", dim_customer, ["customer_id","customer_unique_id"]),
    ("dim_product", dim_product, ["product_id"]),
    ("dim_seller", dim_seller, ["seller_id"]),
    ("fact_orders", fact_orders, ["order_id","customer_id","customer_unique_id"]),
    ("fact_order_items", fact_order_items, ["order_id","order_item_id","product_id","seller_id"]),
    ("fact_payments", fact_payments, ["order_id","payment_sequential"]),
]:
    nulls = df.select([col(c).isNull().cast("int").alias(c) for c in keys]).groupBy().sum().collect()[0]
    print(f"{label}: {dict(zip(keys, nulls))}")

# FK orphans (should be 0)
orphan_orders_customer = fact_orders.select("customer_id").distinct().join(dim_customer.select("customer_id").distinct(), on="customer_id", how="left_anti").count()
orphan_items_product = fact_order_items.select("product_id").distinct().join(dim_product.select("product_id").distinct(), on="product_id", how="left_anti").count()
orphan_items_seller = fact_order_items.select("seller_id").distinct().join(dim_seller.select("seller_id").distinct(), on="seller_id", how="left_anti").count()
orphan_items_order = fact_order_items.select("order_id").distinct().join(fact_orders.select("order_id").distinct(), on="order_id", how="left_anti").count()
orphan_payments_order = fact_payments.select("order_id").distinct().join(fact_orders.select("order_id").distinct(), on="order_id", how="left_anti").count()
print(f"Orphan fact_orders.customer_id not in dim_customer: {orphan_orders_customer}")
print(f"Orphan fact_order_items.product_id not in dim_product: {orphan_items_product}")
print(f"Orphan fact_order_items.seller_id not in dim_seller: {orphan_items_seller}")
print(f"Orphan fact_order_items.order_id not in fact_orders: {orphan_items_order}")
print(f"Orphan fact_payments.order_id not in fact_orders: {orphan_payments_order}")

In [0]:
# Validation 4: Relationship cardinalities & fan-out check
# If facts were incorrectly joined together, row counts would multiply. Verify 1:many holds.
from pyspark.sql import functions as F

print("=== Orders per customer (dim_customer -> fact_orders) ===")
fact_orders.groupBy("customer_unique_id").count().orderBy(F.desc("count")).limit(5).show(truncate=False)

print("=== Items per order (fact_orders -> fact_order_items) ===")
fact_order_items.groupBy("order_id").count().orderBy(F.desc("count")).limit(5).show()

print("=== Payments per order (fact_orders -> fact_payments) ===")
fact_payments.groupBy("order_id").count().orderBy(F.desc("count")).limit(5).show()

# Total additive measure reconciliation (Silver totals should equal Gold totals where no filter)
print("=== Payment value reconciliation ===")
silver_payment_total = payments_silver.agg(F.sum("payment_value")).collect()[0][0]
gold_payment_total = fact_payments.agg(F.sum("payment_value")).collect()[0][0]
print(f"Silver payments sum: {silver_payment_total}  Gold fact_payments sum: {gold_payment_total}  delta={float(gold_payment_total or 0) - float(silver_payment_total or 0)}")

In [0]:
# Validation 5: Schema compliance with locked Gold schemas (Retail_Gold_Model.md #4)
# Lists expected columns; compare with actual Gold schemas
expected_schemas = {
    "dim_customer": ["customer_id","customer_unique_id","customer_zip_code_prefix","customer_city","customer_state"],
    "dim_product": ["product_id","product_category_name","product_category_name_english","product_name_length","product_description_length","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"],
    "dim_seller": ["seller_id","seller_zip_code_prefix","seller_city","seller_state"],
    "fact_orders": ["order_id","customer_unique_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"],
    "fact_order_items": ["order_id","order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value"],
    "fact_payments": ["order_id","payment_sequential","payment_type","payment_installments","payment_value"],
}
for table, expected_cols in expected_schemas.items():
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{table}")
    actual = df.columns
    missing = set(expected_cols) - set(actual)
    extra = set(actual) - set(expected_cols)
    status = "PASS" if not missing and not extra else "FAIL"
    print(f"{table:<20} {status}  missing={missing} extra={extra}")
    df.printSchema()

In [0]:
# Final Gold inventory & documentation ready for Databricks run
spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").show(truncate=False)

for t in ["dim_customer","dim_product","dim_seller","fact_orders","fact_order_items","fact_payments"]:
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{t}")
    print(f"{t:<20} rows={df.count():>8,} cols={len(df.columns)} sample:")
    display(df.limit(5))